# Gün 10-11 — MLflow Deney Takibi

## Adım 2: Her K değeri için ayrı MLflow run

Gün 8-9'da K=2..10 için inertia/silhouette hesapladık ama sonuçlar sadece bu notebook'un içinde, kalıcı/karşılaştırılabilir bir kayıt yoktu. MLflow ile her K denemesini **ayrı bir "run"** olarak loglayıp, ileride "hangi K'da hangi skor vardı" sorusunu kod tekrar çalıştırmadan MLflow UI'dan cevaplayabileceğiz.

In [1]:
import os

import mlflow
import pandas as pd
from dotenv import load_dotenv
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sqlalchemy import create_engine

load_dotenv("../.env")

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
mlflow.set_experiment("rfm-kmeans-segmentation")

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

query = """
    SELECT DISTINCT ON (customer_id) customer_id, recency, frequency, monetary
    FROM segments
    ORDER BY customer_id, calculated_at DESC
"""
rfm = pd.read_sql(query, engine).set_index("customer_id")

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)
print(rfm.shape)

(5878, 3)


`mlflow.set_experiment(...)` aynı işin tüm denemelerini bir "deney" (experiment) altında gruplar — MLflow UI'da farklı projelerin/işlerin run'ları birbirine karışmaz. Şimdi K=2..10 için ayrı run'lar açıp `k` parametresini, `inertia` ve `silhouette` metriklerini logluyoruz.

In [2]:
for k in range(2, 11):
    with mlflow.start_run(run_name=f"kmeans_k{k}"):
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(rfm_scaled)

        mlflow.log_param("k", k)
        mlflow.log_param("random_state", 42)
        mlflow.log_param("n_init", 10)
        mlflow.log_metric("inertia", km.inertia_)
        mlflow.log_metric("silhouette", silhouette_score(rfm_scaled, labels))

print("9 run loglandi (K=2..10).")

🏃 View run kmeans_k2 at: http://127.0.0.1:5000/#/experiments/1/runs/822a799effc246a49b1a7703ab9a0f9f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


🏃 View run kmeans_k3 at: http://127.0.0.1:5000/#/experiments/1/runs/8b70b06d6b9a4c9d8bd609e678565032
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


🏃 View run kmeans_k4 at: http://127.0.0.1:5000/#/experiments/1/runs/e5970947fadc4de884465b5ce285d1da
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


🏃 View run kmeans_k5 at: http://127.0.0.1:5000/#/experiments/1/runs/499e071e5caa4e298d0c0c1e931f67e9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


🏃 View run kmeans_k6 at: http://127.0.0.1:5000/#/experiments/1/runs/87ac3991d11c4a2eb96a4bdaa2f3c9b2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


🏃 View run kmeans_k7 at: http://127.0.0.1:5000/#/experiments/1/runs/3bf59ae6e9474ae79c13962b30368830
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


🏃 View run kmeans_k8 at: http://127.0.0.1:5000/#/experiments/1/runs/0d156c99d6eb47a7bfe7fbb23513e159
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


🏃 View run kmeans_k9 at: http://127.0.0.1:5000/#/experiments/1/runs/95255e55900a47908db1e4e17a84fe5a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


🏃 View run kmeans_k10 at: http://127.0.0.1:5000/#/experiments/1/runs/9ba78f31a15e4ef88b8b97c17efdeb73
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
9 run loglandi (K=2..10).


### Doğrulama

Run'ları kod tekrar çalıştırmadan, MLflow'un kendi API'sinden (`mlflow.search_runs`) çekip görelim — bu, MLflow UI'da (`http://127.0.0.1:5000`) görünenle aynı veri.

In [3]:
runs = mlflow.search_runs(
    experiment_names=["rfm-kmeans-segmentation"],
    order_by=["params.k ASC"],
)
runs[["run_id", "params.k", "metrics.inertia", "metrics.silhouette"]]

,run_id,params.k,metrics.inertia,metrics.silhouette
0,9ba78f31a15e4ef88b8b97c17efdeb73,10,1648.863748,0.500807
1,503c7f297a14463fa677d0dde463481d,10,1648.863748,0.500807
2,187f43dafd094d1e9209b34019505dc2,10,1648.863748,0.500807
3,822a799effc246a49b1a7703ab9a0f9f,2,12114.984358,0.923563
4,8df07d5bba844fe699468e700994c367,2,12114.984358,0.923563
5,1ce73c5608b249fca5ac246b6419b367,2,12114.984358,0.923563
6,8b70b06d6b9a4c9d8bd609e678565032,3,7124.426631,0.580285
7,2b9c41f25b964d0d89de6aed0728f47b,3,7124.426631,0.580285
8,b821099a9d3c4784856c1250fe6ad7cd,3,7124.426631,0.580285
9,e5970947fadc4de884465b5ce285d1da,4,5166.523749,0.591363


## Adım 3: En İyi Modeli Model Registry'ye Kaydetme

Gün 8-9'da K=5'i seçmiştik (silhouette + iş mantığı doğrulamasıyla — sadece "en yüksek skor" değil). Şimdi bu modeli MLflow Model Registry'ye kaydedip "Production" olarak işaretliyoruz.

**Önemli bir nüans:** MLflow 2.9'dan itibaren klasik "stage" (Staging/Production/Archived) sistemi **deprecated** — yeni yaklaşım **alias** + **tag** kullanmak. Görevde istenen "Production etiketi" fikrini hem literal bir **tag** (`stage=Production`) hem de modern, sorgulanabilir bir **alias** (`production`) ile karşılıyoruz — ikisi birbirini tamamlıyor: tag insan-okunur bir etiket, alias ise "şu an production'da çalışan versiyon hangisi" sorusuna API'den cevap veren mekanizma.

In [4]:
import mlflow.sklearn
from sklearn.pipeline import Pipeline

MODEL_NAME = "rfm-customer-segments"
BEST_K = 5

with mlflow.start_run(run_name=f"kmeans_k{BEST_K}_final"):
    final_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("kmeans", KMeans(n_clusters=BEST_K, random_state=42, n_init=10)),
    ])
    final_pipeline.fit(rfm)
    final_labels = final_pipeline.predict(rfm)

    mlflow.log_param("k", BEST_K)
    mlflow.log_metric("inertia", final_pipeline.named_steps["kmeans"].inertia_)
    mlflow.log_metric("silhouette", silhouette_score(rfm_scaled, final_labels))

    model_info = mlflow.sklearn.log_model(
        final_pipeline,
        name="model",
        registered_model_name=MODEL_NAME,
    )

print("Kayitli model versiyonu:", model_info.registered_model_version)

Registered model 'rfm-customer-segments' already exists. Creating a new version of this model...


2026/06/29 19:10:24 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: rfm-customer-segments, version 2


🏃 View run kmeans_k5_final at: http://127.0.0.1:5000/#/experiments/1/runs/581c2f089b844c99862b2c3d6f40342f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Kayitli model versiyonu: 2


Created version '2' of model 'rfm-customer-segments'.


In [5]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
version = model_info.registered_model_version

client.set_model_version_tag(MODEL_NAME, version, "stage", "Production")
client.set_registered_model_alias(MODEL_NAME, "production", version)

mv = client.get_model_version(MODEL_NAME, version)
print("Model:", mv.name, "| Versiyon:", mv.version, "| Tag:", mv.tags)
print("'production' alias'i versiyon", client.get_model_version_by_alias(MODEL_NAME, "production").version, "'e isaretli")

Model: rfm-customer-segments | Versiyon: 2 | Tag: {'stage': 'Production'}
'production' alias'i versiyon 2 'e isaretli


## Adım 4: Artifact Loglama — Segment Profili (CSV) ve Elbow Grafiği (PNG)

Metrikler (sayılar) MLflow'da ayrı bir tabloda tutulur, ama **dosyalar** (CSV, PNG, model dosyası vb.) "artifact" olarak loglanır. Bunları, az önce kaydettiğimiz model versiyonunun **aynı run'ına** ekliyoruz (`run_id` ile yeniden açıyoruz) — böylece "bu model neden seçildi, profili neydi" sorusunun cevabı, modelin kendisiyle birlikte tek bir yerde durur.

In [6]:
import matplotlib.pyplot as plt

segment_profile = rfm.copy()
segment_profile["cluster"] = final_labels
profile_table = segment_profile.groupby("cluster").agg(
    customer_count=("recency", "size"),
    avg_recency=("recency", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary", "mean"),
).round(1).sort_values("avg_monetary", ascending=False)

csv_path = "segment_profile.csv"
profile_table.to_csv(csv_path)
profile_table

,customer_count,avg_recency,avg_frequency,avg_monetary
cluster,,,,
3,4,2.8,212.5,436835.8
2,24,21.8,119.8,100927.0
0,383,27.6,28.5,13935.2
4,3553,75.4,5.1,1911.2
1,1914,471.1,2.2,755.7


In [7]:
elbow_path = "elbow_curve.png"
plt.figure(figsize=(6, 4))
plt.plot(runs.sort_values("params.k", key=lambda s: s.astype(int))["params.k"].astype(int),
          runs.sort_values("params.k", key=lambda s: s.astype(int))["metrics.inertia"], marker="o")
plt.xlabel("K")
plt.ylabel("Inertia (WCSS)")
plt.title("Elbow Curve (MLflow run'larindan)")
plt.savefig(elbow_path, dpi=100)
plt.close()
print("Kaydedildi:", elbow_path)

Kaydedildi: elbow_curve.png


In [8]:
with mlflow.start_run(run_id=model_info.run_id):
    mlflow.log_artifact(csv_path)
    mlflow.log_artifact(elbow_path)

print("Artifact'lar run'a eklendi:", model_info.run_id)

🏃 View run kmeans_k5_final at: http://127.0.0.1:5000/#/experiments/1/runs/581c2f089b844c99862b2c3d6f40342f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Artifact'lar run'a eklendi: 581c2f089b844c99862b2c3d6f40342f


In [9]:
artifacts = client.list_artifacts(model_info.run_id)
[a.path for a in artifacts]

['elbow_curve.png', 'segment_profile.csv']